In [1]:
#pip install pillow numpy opencv-python scikit-image

In [2]:
import json
import shutil
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageDraw
from skimage import measure


In [ ]:
MEDIA_DIR    = Path("./media")       
STATIC_DATA  = Path("../app/static/data")

# ── pipeline params (applied to every image) ─────────────────────────────
chapterCount = 40
chapterWordCounts = [200] * chapterCount

# Normalized field: 0.0 = island edge, 1.0 = canvas border on every side.
minDist      = 0.02
maxDistRatio = 0.90   # stay away from 1.0: gradient is too steep there

minLength        = 2000.0

# Douglas-Peucker simplification: removes staircase pixel-step artifacts.
# At 5000px wide, epsilon=4 is sub-pixel in the final render — keeps shape intact.
simplifyValue    = 4.0

# Laplacian smoothing is DISABLED (0).
# It shrinks nested contours at different rates and causes outer lines to cross.
# The field blur below already smooths paths at the source.
smoothIterations = 0
smoothAlpha      = 0.45
splineSamples    = 1      # Catmull-Rom disabled — overshoots and causes overlap

seaPadding    = 200
borderMargin  = 12
borderPenalty = 0.2

targetLengthPerWord  = 16.0
minLinesPerChapter   = 1
maxLinesPerChapter   = 1

# Blur the normalised distance field before contour extraction.
# This is the ONLY smoothing that respects the nesting order of iso-contours.
# Larger = smoother lines, but loses fine coastal detail.
distFieldBlurSize = 15

blurSize          = 11
saturationWeight  = 0.9
valueWeight       = 0.45
blueWeight        = 1.3
thresholdValue    = 0.95


In [4]:

def ensublackir(pathString):
    Path(pathString).mkdir(parents=True, exist_ok=True)


def loadImage(pathString):
    imageBgr = cv2.imread(str(pathString), cv2.IMREAD_COLOR)
    if imageBgr is None:
        raise FileNotFoundError(f"Could not load image: {pathString}")
    return imageBgr


def buildSeaMaskFromImage(imageBgr, blurSize, saturationWeight, valueWeight, blueWeight, thresholdValue):
    imageHsv = cv2.cvtColor(imageBgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    imageRgb = cv2.cvtColor(imageBgr, cv2.COLOR_BGR2RGB).astype(np.float32)

    hueChannel = imageHsv[:, :, 0] / 179.0
    saturationChannel = imageHsv[:, :, 1] / 255.0
    valueChannel = imageHsv[:, :, 2] / 255.0

    blackChannel = imageRgb[:, :, 0] / 255.0
    greenChannel = imageRgb[:, :, 1] / 255.0
    blueChannel = imageRgb[:, :, 2] / 255.0

    blueCyanScore = np.clip((blueChannel - blackChannel) * 0.7 + (greenChannel - blackChannel) * 0.3, 0.0, 1.0)
    paleSeaScore = np.clip(valueChannel * 0.8 + saturationChannel * 0.2, 0.0, 1.0)
    hueSeaScore = np.exp(-((hueChannel - 0.52) ** 2) / 0.04)

    whitenessScore = valueChannel * np.clip(1.0 - saturationChannel * 4.0, 0.0, 1.0)

    combinedScore = (
        blueWeight * blueCyanScore +
        saturationWeight * saturationChannel +
        valueWeight * paleSeaScore +
        0.6 * hueSeaScore +
        4.0 * whitenessScore
    )

    combinedScore = cv2.GaussianBlur(combinedScore, (blurSize, blurSize), 0)
    seaMask = (combinedScore > thresholdValue).astype(np.uint8) * 255

    kernelSmall = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernelLarge = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

    seaMask = cv2.morphologyEx(seaMask, cv2.MORPH_OPEN, kernelSmall)
    seaMask = cv2.morphologyEx(seaMask, cv2.MORPH_CLOSE, kernelLarge)

    return seaMask


def fillBorderConnectedSea(seaMask):
    height, width = seaMask.shape
    filledMask = seaMask.copy()
    borderConnectedMask = np.zeros_like(seaMask)

    borderSeeds = []
    for x in range(width):
        borderSeeds.append((x, 0))
        borderSeeds.append((x, height - 1))
    for y in range(height):
        borderSeeds.append((0, y))
        borderSeeds.append((width - 1, y))

    for seedX, seedY in borderSeeds:
        if filledMask[seedY, seedX] == 255 and borderConnectedMask[seedY, seedX] == 0:
            tempMask = np.zeros((height + 2, width + 2), np.uint8)
            floodImage = filledMask.copy()
            cv2.floodFill(floodImage, tempMask, (seedX, seedY), 128)
            borderConnectedMask[floodImage == 128] = 255
            filledMask[floodImage == 128] = 0

    return borderConnectedMask


def buildSeaMask(imageBgr, manualMaskPath, blurSize, saturationWeight, valueWeight, blueWeight, thresholdValue):
    if manualMaskPath is not None:
        maskGray = cv2.imread(str(manualMaskPath), cv2.IMREAD_GRAYSCALE)
        if maskGray is None:
            raise FileNotFoundError(f"Could not load manual mask: {manualMaskPath}")
        return (maskGray > 127).astype(np.uint8) * 255

    autoMask = buildSeaMaskFromImage(
        imageBgr=imageBgr,
        blurSize=blurSize,
        saturationWeight=saturationWeight,
        valueWeight=valueWeight,
        blueWeight=blueWeight,
        thresholdValue=thresholdValue
    )

    return fillBorderConnectedSea(autoMask)


def buildNormalizedField(seaMask):
    """Normalized potential field: 0.0 at island edge, 1.0 at canvas border."""
    seaBinary = (seaMask > 0).astype(np.uint8)
    distIsland = cv2.distanceTransform(seaBinary, cv2.DIST_L2, 5)
    interior = np.ones(seaMask.shape, dtype=np.uint8)
    interior[0, :] = interior[-1, :] = interior[:, 0] = interior[:, -1] = 0
    distBorder = cv2.distanceTransform(interior, cv2.DIST_L2, 5)
    normalized = distIsland / (distIsland + distBorder + 1e-6)
    normalized[seaBinary == 0] = 0.0
    return normalized.astype(np.float32)


def simplifyPolyline(pointsArray, epsilonValue, closed=False):
    if len(pointsArray) < 3:
        return pointsArray
    pointsFloat = pointsArray.astype(np.float32).reshape((-1, 1, 2))
    simplified = cv2.approxPolyDP(pointsFloat, epsilonValue, closed)
    return simplified.reshape((-1, 2)).astype(np.float32)


def smoothPolyline(pointsArray, iterations, alpha):
    if len(pointsArray) < 3:
        return pointsArray
    pts = pointsArray.copy()
    for _ in range(iterations):
        newPts = pts.copy()
        for i in range(1, len(pts) - 1):
            newPts[i] = pts[i] * (1 - alpha) + (pts[i - 1] + pts[i + 1]) * (alpha * 0.5)
        pts = newPts
    return pts.astype(np.float32)


def catmullRomSpline(pointsArray, samplesPerSegment):
    if len(pointsArray) < 2:
        return pointsArray

    result = []
    pointCount = len(pointsArray)
    isClosed = np.linalg.norm(pointsArray[0] - pointsArray[-1]) < 1.0

    for i in range(pointCount - 1):
        if isClosed:
            p0 = pointsArray[i - 1] if i > 0 else pointsArray[-2]
            p3 = pointsArray[i + 2] if i + 2 < pointCount else pointsArray[i + 2 - (pointCount - 1)]
        else:
            p0 = pointsArray[i - 1] if i > 0 else pointsArray[i]
            p3 = pointsArray[i + 2] if i + 2 < pointCount else pointsArray[i + 1]
        p1 = pointsArray[i]
        p2 = pointsArray[i + 1]

        for t in np.linspace(0, 1, samplesPerSegment, endpoint=False):
            t2 = t * t
            t3 = t2 * t
            x = 0.5 * ((2*p1[0]) + (-p0[0]+p2[0])*t + (2*p0[0]-5*p1[0]+4*p2[0]-p3[0])*t2 + (-p0[0]+3*p1[0]-3*p2[0]+p3[0])*t3)
            y = 0.5 * ((2*p1[1]) + (-p0[1]+p2[1])*t + (2*p0[1]-5*p1[1]+4*p2[1]-p3[1])*t2 + (-p0[1]+3*p1[1]-3*p2[1]+p3[1])*t3)
            result.append([x, y])

    result.append(pointsArray[-1].tolist())
    return np.array(result, dtype=np.float32)


def contourLength(pointsArray):
    if len(pointsArray) < 2:
        return 0.0
    diffs = np.diff(pointsArray, axis=0)
    return float(np.sum(np.sqrt(np.sum(diffs * diffs, axis=1))))


def touchesBorder(pointsArray, imageWidth, imageHeight, margin):
    xs = pointsArray[:, 0]
    ys = pointsArray[:, 1]
    return bool(
        np.any(xs <= margin) or
        np.any(xs >= imageWidth - 1 - margin) or
        np.any(ys <= margin) or
        np.any(ys >= imageHeight - 1 - margin)
    )


def buildLevels(distanceField, chapterCount, minDist, maxDistRatio):
    fieldMax = float(np.max(distanceField))
    maxDist = fieldMax * maxDistRatio
    if maxDist <= minDist:
        maxDist = fieldMax
    return np.linspace(minDist, maxDist, chapterCount)


def stitchFragments(fragments, maxGap=50.0):
    if len(fragments) <= 1:
        return fragments
    chains = [f.tolist() for f in fragments]
    changed = True
    while changed:
        changed = False
        bestDist = maxGap
        bestMerge = None
        for i in range(len(chains)):
            for j in range(i + 1, len(chains)):
                tests = [
                    (chains[i][-1], chains[j][0],  False, False),
                    (chains[i][-1], chains[j][-1], False, True),
                    (chains[i][0],  chains[j][0],  True,  False),
                    (chains[i][0],  chains[j][-1], True,  True),
                ]
                for ei, ej, flipI, flipJ in tests:
                    d = ((ei[0]-ej[0])**2 + (ei[1]-ej[1])**2) ** 0.5
                    if d < bestDist:
                        bestDist = d
                        bestMerge = (i, j, flipI, flipJ)
        if bestMerge:
            i, j, flipI, flipJ = bestMerge
            ci = list(reversed(chains[i])) if flipI else chains[i]
            cj = list(reversed(chains[j])) if flipJ else chains[j]
            chains[i] = ci + cj
            chains.pop(j)
            changed = True
    return [np.array(c, dtype=np.float32) for c in chains]


def rotateToSmoothestSeam(pts):
    """Rotate a closed polyline so its seam sits at the point of minimum curvature.

    At the straightest point the incoming and outgoing tangents are nearly equal,
    so the LUT discontinuity is minimal and animated words crossing it won't glitch.
    pts[-1] is expected to equal pts[0] (closed loop).
    """
    closed = np.linalg.norm(pts[-1] - pts[0]) < 0.5
    ring = pts[:-1] if closed else pts
    m = len(ring)
    if m < 4:
        return pts

    best_idx = 0
    min_curvature = float('inf')

    for i in range(m):
        prev = ring[(i - 1) % m]
        curr = ring[i]
        nxt  = ring[(i + 1) % m]
        d_in  = curr - prev
        d_out = nxt  - curr
        l_in  = np.linalg.norm(d_in)
        l_out = np.linalg.norm(d_out)
        if l_in < 1e-6 or l_out < 1e-6:
            continue
        cos_a = np.dot(d_in / l_in, d_out / l_out)
        curvature = 1.0 - float(np.clip(cos_a, -1.0, 1.0))  # 0=straight, 2=U-turn
        if curvature < min_curvature:
            min_curvature = curvature
            best_idx = i

    rotated = np.vstack([ring[best_idx:], ring[:best_idx]])
    if closed:
        rotated = np.vstack([rotated, rotated[0:1]])
    return rotated.astype(np.float32)


def extractContours(distanceField, levels, minLength, simplifyValue, smoothIterations, smoothAlpha, splineSamples, imageWidth, imageHeight, borderMargin):
    contourItems = []

    for levelIndex, levelValue in enumerate(levels):
        rawContours = measure.find_contours(distanceField, float(levelValue))

        fragments = []
        for contour in rawContours:
            if len(contour) < 2:
                continue
            xy = np.stack([contour[:, 1], contour[:, 0]], axis=1).astype(np.float32)
            fragments.append(xy)

        stitched = stitchFragments(fragments, maxGap=200.0)

        for contourIndex, contourXy in enumerate(stitched):
            endGap = np.linalg.norm(contourXy[-1] - contourXy[0])
            rawLen = float(np.sum(np.linalg.norm(np.diff(contourXy, axis=0), axis=1)))
            isLikelyClosed = (endGap < 80.0 or endGap < rawLen * 0.10) and endGap < rawLen * 0.5

            if simplifyValue > 0:
                contourXy = simplifyPolyline(contourXy, simplifyValue, closed=isLikelyClosed)

            if len(contourXy) < 2:
                continue

            contourXy = smoothPolyline(contourXy, smoothIterations, smoothAlpha)

            if isLikelyClosed:
                gap2 = np.linalg.norm(contourXy[-1] - contourXy[0])
                if gap2 > 0.5:
                    contourXy = np.vstack([contourXy, contourXy[0:1]])
                # Rotate so the seam falls at the straightest point of the curve
                contourXy = rotateToSmoothestSeam(contourXy)

            if splineSamples > 1:
                contourXy = catmullRomSpline(contourXy, splineSamples)

            if len(contourXy) < 2:
                continue

            lengthValue = contourLength(contourXy)
            if lengthValue < minLength:
                continue

            contourItems.append({
                "id": f"contour_{levelIndex:02d}_{contourIndex:04d}",
                "levelIndex": int(levelIndex),
                "level": float(levelValue),
                "length": float(lengthValue),
                "touchesBorder": touchesBorder(contourXy, imageWidth, imageHeight, borderMargin),
                "points": contourXy.tolist()
            })

    return contourItems


def scoreContour(contourItem, borderPenalty):
    edgeFactor = borderPenalty if contourItem["touchesBorder"] else 1.0
    return contourItem["length"] * edgeFactor


def sortContoursSpatially(contourItems, borderPenalty):
    rankedContours = sorted(contourItems, key=lambda item: (item["levelIndex"], -scoreContour(item, borderPenalty)))
    rankedContours = sorted(rankedContours, key=lambda item: item["levelIndex"])
    return rankedContours


def assignContoursSequentially(sortedContours, levels, chapterWordCounts, targetLengthPerWord, minLinesPerChapter, maxLinesPerChapter):
    from collections import defaultdict
    byLevel = defaultdict(list)
    for c in sortedContours:
        byLevel[c["levelIndex"]].append(c)

    numLevels = len(levels)
    chapterItems = []

    for chapterIndex in range(numLevels):
        chapterNumber = numLevels - chapterIndex
        contoursAtLevel = byLevel.get(chapterIndex, [])
        contoursAtLevel = sorted(contoursAtLevel, key=lambda c: (c["touchesBorder"], -c["length"]))
        contoursAtLevel = contoursAtLevel[:maxLinesPerChapter]

        selectedContours = []
        for i, contourItem in enumerate(contoursAtLevel):
            selectedContours.append({
                "id": f"chapter_{chapterNumber:02d}_line_{i + 1:02d}",
                "sourceId": contourItem["id"],
                "chapter": int(chapterNumber),
                "lineIndex": int(i + 1),
                "targetLevelIndex": int(chapterIndex),
                "sourceLevelIndex": int(contourItem["levelIndex"]),
                "level": float(levels[chapterIndex]),
                "sourceLevel": float(contourItem["level"]),
                "length": float(contourItem["length"]),
                "touchesBorder": bool(contourItem["touchesBorder"]),
                "points": contourItem["points"],
            })

        totalLength = sum(c["length"] for c in selectedContours)
        chapterItems.append({
            "chapter": int(chapterNumber),
            "wordCount": int(chapterWordCounts[chapterIndex] if chapterIndex < len(chapterWordCounts) else 0),
            "targetLength": float(0),
            "totalLength": float(totalLength),
            "lineCount": int(len(selectedContours)),
            "lineIds": [item["id"] for item in selectedContours],
            "lines": selectedContours,
        })

    return chapterItems


def flattenChapterLines(chapterItems):
    sortedItems = sorted(chapterItems, key=lambda ci: ci["chapter"])
    flattened = []
    for chapterItem in sortedItems:
        lines = sorted(chapterItem["lines"], key=lambda item: (item["touchesBorder"], -item["length"]))
        flattened.extend(lines)
    for i, line in enumerate(flattened):
        line["pathOrder"] = i
    return flattened


def shiftContourPoints(contourLines, dx, dy):
    for line in contourLines:
        line["points"] = [[x + dx, y + dy] for x, y in line["points"]]


def saveJson(chapterItems, contourLines, outputPath, imageWidth, imageHeight, levels):
    payload = {
        "width": imageWidth,
        "height": imageHeight,
        "chapterCount": len(chapterItems),
        "levels": [float(levelValue) for levelValue in levels],
        "chapters": chapterItems,
        "lines": contourLines
    }
    with open(outputPath, "w", encoding="utf-8") as fileHandle:
        json.dump(payload, fileHandle, ensure_ascii=False, indent=2)


def polylineToSvgPath(pointsList):
    if not pointsList:
        return ""
    firstX, firstY = pointsList[0]
    parts = [f"M {firstX:.2f} {firstY:.2f}"]
    for pointX, pointY in pointsList[1:]:
        parts.append(f"L {pointX:.2f} {pointY:.2f}")
    parts.append("Z")
    return " ".join(parts)


def saveSvg(contourLines, outputPath, imageWidth, imageHeight):
    svgParts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {imageWidth} {imageHeight}" width="{imageWidth}" height="{imageHeight}">',
        '<g fill="none" stroke="black" stroke-width="1">',
    ]
    for lineItem in contourLines:
        pathData = polylineToSvgPath(lineItem["points"])
        svgParts.append(
            f'<path id="{lineItem["id"]}" data-chapter="{lineItem["chapter"]}" d="{pathData}" />'
        )
    svgParts += ["</g>", "</svg>"]
    with open(outputPath, "w", encoding="utf-8") as fileHandle:
        fileHandle.write("\n".join(svgParts))


def saveMask(maskArray, outputPath):
    Image.fromarray(maskArray).save(outputPath)


def drawPreview(imageBgr, contourLines, outputPath):
    imageRgb = cv2.cvtColor(imageBgr, cv2.COLOR_BGR2RGB)
    previewImage = Image.fromarray(imageRgb)
    drawLayer = ImageDraw.Draw(previewImage)
    for lineItem in contourLines:
        pointsTuple = [tuple(point) for point in lineItem["points"]]
        if len(pointsTuple) >= 2:
            drawLayer.line(pointsTuple, fill=(30, 60, 90), width=2)
    previewImage.save(outputPath)


def runPipelineForImage(imagePath, maskPath, suffix):
    imageBgr = loadImage(imagePath)
    origH, origW = imageBgr.shape[:2]

    seaMask = buildSeaMask(
        imageBgr=imageBgr,
        manualMaskPath=maskPath,
        blurSize=blurSize,
        saturationWeight=saturationWeight,
        valueWeight=valueWeight,
        blueWeight=blueWeight,
        thresholdValue=thresholdValue
    )

    pad = seaPadding
    seaMaskPadded = np.pad(seaMask, pad, mode="constant", constant_values=255)
    paddedH, paddedW = seaMaskPadded.shape

    distanceField = buildNormalizedField(seaMaskPadded)

    if distFieldBlurSize > 1:
        ksize = distFieldBlurSize if distFieldBlurSize % 2 == 1 else distFieldBlurSize + 1
        distanceField = cv2.GaussianBlur(distanceField, (ksize, ksize), 0)

    levels = buildLevels(distanceField, chapterCount, minDist, maxDistRatio)

    contourItems = extractContours(
        distanceField=distanceField,
        levels=levels,
        minLength=minLength,
        simplifyValue=simplifyValue,
        smoothIterations=smoothIterations,
        smoothAlpha=smoothAlpha,
        splineSamples=splineSamples,
        imageWidth=paddedW,
        imageHeight=paddedH,
        borderMargin=borderMargin
    )

    sortedContours = sortContoursSpatially(contourItems, borderPenalty)

    chapterItems = assignContoursSequentially(
        sortedContours=sortedContours,
        levels=levels,
        chapterWordCounts=chapterWordCounts,
        targetLengthPerWord=targetLengthPerWord,
        minLinesPerChapter=minLinesPerChapter,
        maxLinesPerChapter=maxLinesPerChapter
    )

    contourLines = flattenChapterLines(chapterItems)
    shiftContourPoints(contourLines, dx=-pad, dy=-pad)

    ensublackir(str(STATIC_DATA))

    destImg = STATIC_DATA / f"image{suffix}.jpg"
    img_rgb = cv2.cvtColor(imageBgr, cv2.COLOR_BGR2RGB)
    Image.fromarray(img_rgb).save(str(destImg), "JPEG", quality=92)

    contoursDest = STATIC_DATA / f"contours{suffix}.json"
    saveJson(chapterItems, contourLines, contoursDest, origW, origH, levels)

    previewDir = Path(f"preview{suffix}")
    ensublackir(str(previewDir))
    saveMask(seaMask, previewDir / "seaMask.png")
    saveSvg(contourLines, previewDir / "contours.svg", origW, origH)
    drawPreview(imageBgr, contourLines, previewDir / "preview.png")

    return {
        "suffix": suffix,
        "imageSrc": str(imagePath),
        "imageWidth": origW,
        "imageHeight": origH,
        "generatedLines": len(contourLines),
        "maskUsed": str(maskPath) if maskPath else "auto",
    }


In [5]:
# ── Batch: process all images in ./media/ ────────────────────────────────
# Naming convention:
#   media/foo.jpg            → contours.json  / image.jpg   (dataset 0 / "default")
#   media/bar.jpg            → contours1.json / image1.jpg  (dataset 1)
#   …sorted alphabetically…
#
# Masks: place an optional mask next to each image as <stem>_mask.png
#   e.g.  media/foo_mask.png   is loaded automatically for media/foo.jpg
# If no mask file is found, the sea is detected automatically from colour.

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

images = sorted(
    p for p in MEDIA_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTS and "_mask" not in p.stem
)

if not images:
    raise FileNotFoundError(f"No images found in {MEDIA_DIR.resolve()}")

results = []
for idx, imgPath in enumerate(images):
    suffix = "" if idx == 0 else str(idx)
    # look for <stem>_mask.png / <stem>_mask.jpg next to the image
    maskPath = None
    for ext in [".png", ".jpg", ".jpeg"]:
        candidate = imgPath.parent / (imgPath.stem + "_mask" + ext)
        if candidate.exists():
            maskPath = candidate
            break

    print(f"[{idx}] {imgPath.name}  mask={maskPath.name if maskPath else 'auto'}  → contours{suffix}.json")
    result = runPipelineForImage(imgPath, maskPath, suffix)
    results.append(result)
    print(f"     {result['imageWidth']}×{result['imageHeight']}  {result['generatedLines']} lines")

print("\nDone. Files written to", STATIC_DATA.resolve())
results


[0] image001.jpg  mask=image001_mask.png  → contours.json
     5000×6667  40 lines
[1] image002.jpg  mask=image002_mask.png  → contours1.json
     5000×6667  40 lines
[2] image01.jpg  mask=image01_mask.png  → contours2.json
     5000×6667  40 lines
[3] image04.jpg  mask=image04_mask.png  → contours3.json
     5000×6667  40 lines
[4] image05.jpg  mask=image05_mask.png  → contours4.json
     5000×6667  40 lines
[5] image06.jpg  mask=image06_mask.png  → contours5.json
     5000×6667  40 lines
[6] image07.jpg  mask=image07_mask.png  → contours6.json
     5000×6667  40 lines
[7] image08.jpg  mask=image08_mask.png  → contours7.json
     5000×6667  40 lines
[8] image09.jpg  mask=image09_mask.png  → contours8.json
     5000×6667  40 lines
[9] image10.jpg  mask=image10_mask.png  → contours9.json
     5000×6667  40 lines
[10] image11.jpg  mask=image11_mask.png  → contours10.json
     5000×6667  40 lines
[11] image12.jpg  mask=image12_mask.png  → contours11.json
     5000×6667  40 lines
[12] imag

[{'suffix': '',
  'imageSrc': 'media/image001.jpg',
  'imageWidth': 5000,
  'imageHeight': 6667,
  'generatedLines': 40,
  'maskUsed': 'media/image001_mask.png'},
 {'suffix': '1',
  'imageSrc': 'media/image002.jpg',
  'imageWidth': 5000,
  'imageHeight': 6667,
  'generatedLines': 40,
  'maskUsed': 'media/image002_mask.png'},
 {'suffix': '2',
  'imageSrc': 'media/image01.jpg',
  'imageWidth': 5000,
  'imageHeight': 6667,
  'generatedLines': 40,
  'maskUsed': 'media/image01_mask.png'},
 {'suffix': '3',
  'imageSrc': 'media/image04.jpg',
  'imageWidth': 5000,
  'imageHeight': 6667,
  'generatedLines': 40,
  'maskUsed': 'media/image04_mask.png'},
 {'suffix': '4',
  'imageSrc': 'media/image05.jpg',
  'imageWidth': 5000,
  'imageHeight': 6667,
  'generatedLines': 40,
  'maskUsed': 'media/image05_mask.png'},
 {'suffix': '5',
  'imageSrc': 'media/image06.jpg',
  'imageWidth': 5000,
  'imageHeight': 6667,
  'generatedLines': 40,
  'maskUsed': 'media/image06_mask.png'},
 {'suffix': '6',
  'image